# Big Data Analytics for Intelligent Resume Screening

## 1. Introduction & Abstract
This project tackles the Big Data challenge of processing unstructured resume data at volume and velocity. We propose a Machine Learning (ML) pipeline leveraging Apache Spark (PySpark) to classify resumes into 24 distinct job categories.

## 2. Experimental Setup and PySpark Initialization
Initializing the Spark Session for distributed processing.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim, length

spark = SparkSession.builder \
    .appName("Resume_Classification_BDA") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="whitegrid")

## 3. Dataset & Pre-processing (ETL Workflow)
Loading the Resume dataset (CSV) containing unstructured 'Resume_str' and structured 'Category' labels.

In [ ]:
data_path = "dataset/Resume/Resume.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True, escape='"')

print(f"Total Records: {df.count()}")
df.printSchema()
df.select("ID", "Category").show(5)

In [ ]:
# Drop rows with nulls in critical columns
df = df.dropna(subset=["Resume_str", "Category"])

# Clean text: Lowercase, remove special characters, trim whitespace
df_clean = df.withColumn("clean_resume", lower(col("Resume_str")))
df_clean = df_clean.withColumn("clean_resume", regexp_replace(col("clean_resume"), r"[^a-zA-Z0-9\s]", " "))
df_clean = df_clean.withColumn("clean_resume", trim(regexp_replace(col("clean_resume"), r"\s+", " ")))

df_clean.select("Category", "clean_resume").show(3, truncate=80)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Category distribution
category_counts = df_clean.groupBy("Category").count().orderBy(col("count").desc()).toPandas()

plt.figure(figsize=(14, 6))
sns.barplot(data=category_counts, x="Category", y="count", palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Distribution of Resume Categories")
plt.tight_layout()
plt.show()

## 5. Feature Engineering: NLP Pipeline
Using Spark MLLib for Tokenization, StopWordsRemover, and HashingTF/IDF to convert text into numeric vectors.

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml import Pipeline

# 1. Label Indexing
indexer = StringIndexer(inputCol="Category", outputCol="label")

# 2. Tokenization
tokenizer = Tokenizer(inputCol="clean_resume", outputCol="words")

# 3. Stop words removal
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# 4. Term Frequency (TF)
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=1000)

# 5. Inverse Document Frequency (IDF)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# Build pipeline
nlp_pipeline = Pipeline(stages=[indexer, tokenizer, remover, hashingTF, idf])

# Fit & Transform
nlp_model = nlp_pipeline.fit(df_clean)
dataset = nlp_model.transform(df_clean)

dataset.select("label", "features").show(5)

## 6. Model Training & Distributed ML Workflow
We use Logistic Regression and Naive Bayes to perform multi-class classification on the extracted TF-IDF features.

In [ ]:
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 70/30 Train/Test Split
train_data, test_data = dataset.randomSplit([0.7, 0.3], seed=42)
print(f"Training Data: {train_data.count()}")
print(f"Testing Data: {test_data.count()}")

# Logistic Regression Classifier
lr = LogisticRegression(maxIter=10, regParam=0.01)
lr_model = lr.fit(train_data)

# Predict on Test Data
predictions = lr_model.transform(test_data)

## 7. Results & Performance Metrics
Evaluating Accuracy, Precision, Recall, and F1-Score.

In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="f1")
evaluator_precision = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="weightedRecall")

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)
precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1_score:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

## 8. Social Impact Analysis & Conclusion
- **Scalability Analysis**: The PySpark pipeline demonstrates how data volume can scale seamlessly using distributed processing overheads, enabling larger organizational dataset parsing.
- **Ethical Implications**: In Intelligent Recruiting and HR analytics, minimizing human-bias through structured classification serves as a social good equalizer.
- **Future Scope**: Implementation of Quantum-inspired Optimization for Hashing and distributed training over blockchain consensus models for verified credentials.